# CS 6320 Project 3: Projection Matrix and Fundamental Matrix Estimation

This notebook is your report. You are expected to fill it in with code, figures, and written
answers. **Run every cell from top to bottom before you submit** — anything that is not visible in
the saved notebook cannot be graded.


The project has these parts:

1. **Projection matrix estimation (49 pts)**: project 3D world points into an image, recover the
   camera matrix $\mathbf{P}$ from 2D–3D correspondences, and decompose it into intrinsics,
   rotation, and camera center.
2. **Fundamental matrix estimation (51 pts)**: recover the epipolar geometry relating two views of
   the same scene, and draw the epipolar lines.
3. **Robust estimation with RANSAC (10 pts extra credit)**: handle a set of automatic feature
   matches where a third of them are wrong.

Total: **100 points**, plus 10 points of extra credit and a required AI-use disclosure at the end.




## Brief

- **Due:** see Canvas
- **Hand in:** through Gradescope
- **Required file:** `<your_uid>.zip` (your UID starts with `u`, e.g. `u1234567.zip`)

## Setup

0. Unzip the project and enter its folder: `unzip proj3_6320.zip && cd proj3_6320`
1. Install [Miniconda](https://docs.conda.io/en/latest/miniconda.html) if you do not already have
   conda.
2. Create the environment, substituting your OS (`linux`, `mac`, or `win`):
   `conda env create -f proj3_env_<OS>.yml`. The `.yml` files are at the top level of the project
   folder. On Windows run this from the "Anaconda Prompt"; on macOS/Linux any terminal works.
3. Activate it: `conda activate proj3`
4. Install the project package from the project folder: `pip install -e .`
5. Start Jupyter from the project folder (`jupyter notebook`) and open
   `proj3_code/proj3.ipynb`. **Run it from `proj3_code/`**, every path in this notebook is
   relative to that directory (`../data/`, `../results/`), so moving the notebook breaks it.
6. Check your work as you go with the `verify(...)` calls built into the notebook, or run the whole
   suite at once from the project folder with `pytest proj3_code/unit_tests`.
7. When you are done, run every cell one last time, save, and build your submission with
   `python zip_submission.py --uid <your_uid>`.

### What you will need to supply yourself

Parts 1.6–1.8 and 2.2 ask you to **photograph your own scene**. Take those photos early, you cannot do at the last minute:

- **Parts 1.6–1.8** need two photos of a *fiducial object*: something rigid whose dimensions you can
  measure with a ruler. A thick textbook or a cardboard box works well. Put it against a plain,
  contrasting background.
- **Part 2.2** needs two photos of a scene with plenty of texture at a **range of depths**, taken
  from two positions that differ by a real **translation** (not just a rotation or a zoom. Question
  2.3.1 asks you why).

Save them in `../data/` and point the notebook at them. Two example photographs
(`new_book_img1.jpg`, `new_book_img2.jpg`) ship with the project so that every cell runs before you
have your own; they show the kind of framing that works. **You are graded on your own images**, so
replace the filenames once you have them.

## Writeup

Answer every question in the markdown cell provided for it. Do not reorder cells; you may add
cells wherever you need them. Output images are written to `../results/`, which already exists.

## Rubric

**Part 1 — Projection matrix (49 points)**

| Section | Points | What is graded |
| --- | --- | --- |
| 1.1 | 6 | `projection()` |
| 1.2 | 6 | `objective_func()` |
| 1.3 | 6 | `estimate_camera_matrix()` |
| 1.4 | 6 | `decompose_camera_matrix()` |
| 1.5 | 6 | `calculate_camera_center()` |
| 1.6 | 5 | Your own images, 3D measurements, and 2D correspondences |
| 1.7 | 5 | Your own $\mathbf{K}$, $\mathbf{R}$, $\mathbf{t}$ initial estimates, and the resulting fit |
| 1.8 | 4 | Both camera poses visualized in the world frame |
| 1.9 | 5 | The two camera-center experiments and your written answers |

**Part 2 — Fundamental matrix (51 points)**

| Section | Points | What is graded |
| --- | --- | --- |
| 2.1 | 20 | `point_line_distance()`, `signed_point_line_errors()`, `optimize()`, and the epipolar lines on the provided pair |
| 2.2 | 15 | Your own image pair, correspondences, and epipolar lines |
| 2.3 | 16 | Six written questions (3 pts each for 1–4, 2 pts each for 5–6) |
| 2.4 | **+10** | **Extra credit** — RANSAC: 2.4.1 (2), 2.4.2 (2), 2.4.3 (2), 2.4.4 (4) |

Parts 1 and 2.1–2.3 total 100 points. **Part 2.4 is extra credit on top of that**, so the maximum
possible score is 110. Leave it until the rest works: it reuses `solve_F` from Part 2.1 and builds
on Question 2.3.6.

**Part 3 — required, 0 points.** Complete the AI-use disclosure at the end of the notebook,
including if you used no AI tools.

**Deductions.** −5 points each time the hand-in instructions are not followed (wrong file name,
missing outputs, absolute paths, notebook moved out of `proj3_code/`, missing AI-use disclosure).

### Submission contents

`<your_uid>.zip`, produced by `zip_submission.py`, containing `proj3_code/` (your code and the run
notebook), `data/` (including the images you took), and `results/`. Please stay inside the provided
conda environment and use relative paths only.


## Setup

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os

import cv2
import numpy as np
import matplotlib.pyplot as plt

from utils import *

RESULTS = '../results'
os.makedirs(RESULTS, exist_ok=True)

np.set_printoptions(suppress=True, precision=6)


# Part 1: Projection Matrix Estimation

A pinhole camera maps a 3D world point $\mathbf{X}_w$ to an image point $\mathbf{x}$ through a
single $3 \times 4$ matrix:

$$\mathbf{x} \sim \mathbf{P}\,\mathbf{X}_w, \qquad
  \mathbf{P} = \mathbf{K}\begin{bmatrix}\mathbf{R} & \mathbf{t}\end{bmatrix}$$

where $\mathbf{K}$ holds the intrinsics (focal length, principal point) and
$[\mathbf{R}\;|\;\mathbf{t}]$ the pose of the camera in the world. The $\sim$ matters: the
product is a homogeneous 3-vector, so you have to divide by its third entry to get pixels, and
$\mathbf{P}$ is only defined up to scale. We pin that scale down by fixing $P_{34} = 1$, which is
why you will optimize over 11 numbers rather than 12.

In this part you will implement each step: project points with a known $\mathbf{P}$, estimate
$\mathbf{P}$ from correspondences, then pull $\mathbf{K}$, $\mathbf{R}$, and the camera center
back out of it.


## Part 1.1: Implement camera projection (6 pts)

In `projection_matrix.py`, implement `projection(P, points_3d)`, which maps homogeneous world
coordinates $\mathbf{X}_i = [X_i, Y_i, Z_i, 1]$ to non-homogeneous pixel coordinates $(x_i, y_i)$:

\begin{align}
x_i = \frac{p_{11}X_i+p_{12}Y_i + p_{13}Z_i + p_{14}}{p_{31}X_i+p_{32}Y_i + p_{33}Z_i + p_{34}}
\quad
y_i = \frac{p_{21}X_i+p_{22}Y_i + p_{23}Z_i + p_{24}}{p_{31}X_i+p_{32}Y_i + p_{33}Z_i + p_{34}}
\end{align}

Accept `points_3d` as either $n \times 3$ or $n \times 4$ (append the column of ones yourself if
it is $n \times 3$) and return an $n \times 2$ array. Do not forget the division by the third row, without it you have homogeneous coordinates, not pixels.



In [2]:
import projection_matrix

from unit_tests.part1_unit_test import (
    verify,
    test_projection,
    test_objective_func,
    test_decompose_camera_matrix,
    test_calculate_camera_center,
    test_estimate_camera_matrix,
)

Each `verify(...)` call below prints `"Correct"` in green or `"Wrong"` in red.
`"Wrong"` means an assertion failed; run the test function directly (e.g. `test_projection()`) to
see the actual assertion message, which names what mismatched.

In [3]:
print('Test for camera projection:', verify(test_projection))

Test for camera projection: "Correct"


## Part 1.2: Objective function (6 pts)

Implement `objective_func(x, **kwargs)` in `projection_matrix.py`. This is the function handed to
`scipy.optimize.least_squares`, which minimizes the sum of squared reprojection errors:

\begin{align}
\sum_{i=1}^N \left( \color{purple}{\hat{\mathbf{P}}\mathbf{X}_w^i-\mathbf{x}^i} \right)^2
\end{align}

SciPy does the squaring and summing, so return the **unsquared, flattened** residuals: a vector of
length $2N$ holding every $x$ difference and every $y$ difference.

- `x` is the camera matrix flattened to 11 parameters, with $P_{34}$ fixed at 1. Reshape it back to
  $3 \times 4$ by appending that 1.
- The 2D and 3D points arrive through `kwargs`, under the keys `'pts2d'` and `'pts3d'`.


In [15]:
print('Test for objective_function:', verify(test_objective_func))

Test for objective_function: "Correct"


## Part 1.3: Estimating the projection matrix from correspondences (6 pts)

Implement `estimate_camera_matrix(pts2d, pts3d, initial_guess)` in `projection_matrix.py`. It should
call `scipy.optimize.least_squares` with your objective function and return the recovered
$3 \times 4$ matrix.

### Why the initial guess matters

Optimizing the reprojection loss using Levenberg-Marquardt requires a good initial estimate for $\mathbf{P}$. From a good starting point it converges in a
fraction of a second; from a bad one it can wander for 50,000 function evaluations and return
nonsense. For this first run we give you a working initial guess, built from rough estimates of
$\mathbf{K}$, $\mathbf{R}^T$, and $[\mathbf{I}|\mathbf{t}]$. In Part 1.7 you will have to
construct your own, and you will see exactly how unforgiving this is.


In [ ]:
# A working initial guess, provided for this first run.
initial_guess_K = np.array([[500,   0, 535],
                            [  0, 500, 390],
                            [  0,   0,  -1]])

initial_guess_R_T = np.array([[0.5,  -1,  0],
                              [  0,   0, -1],
                              [  1, 0.5,  0]])

initial_guess_I_t = np.array([[1, 0, 0, 300],
                              [0, 1, 0, 300],
                              [0, 0, 1,  30]])

initial_guess_P = np.matmul(initial_guess_K, np.matmul(initial_guess_R_T, initial_guess_I_t))


In [ ]:
# Load the provided correspondences
pts2d_path = '../data/pts2d-pic_b.txt'
pts3d_path = '../data/pts3d.txt'
img_path = '../data/pic_b.jpg'

points_2d = np.loadtxt(pts2d_path)
points_3d = np.loadtxt(pts3d_path)
img = load_image(img_path)

print(f'{len(points_2d)} correspondences, image {img.shape[1]} x {img.shape[0]} px')


### Estimate the projection matrix given corresponding 2D and 3D points

In [ ]:
print('Test for estimate_camera_matrix:', verify(test_estimate_camera_matrix))


In [ ]:
P = projection_matrix.estimate_camera_matrix(points_2d, points_3d, initial_guess_P)

print('The projection matrix is\n', P)

projected_2d_pts, residual = evaluate_points(P, points_2d, points_3d)

# residual is the SUM of Euclidean distances over all points, so divide by the point
# count to get the per-point error that is easier to reason about.
print('Total residual: {:.4f} px over {} points ({:.4f} px per point)'.format(
    residual, len(points_2d), residual / len(points_2d)))

visualize_points_image(points_2d, projected_2d_pts, img_path)


## Part 1.4: Decomposing the projection matrix (6 pts)

Implement `decompose_camera_matrix(P)`, which splits the camera matrix into the intrinsic matrix
$\mathbf{K}$ and the rotation ${}_c\mathbf{R}_w$ (world-to-camera).

The left $3 \times 3$ block of $\mathbf{P}$ equals $\mathbf{K}\mathbf{R}$, a product of an
upper-triangular matrix and an orthonormal one. That is exactly an **RQ decomposition**, so
`scipy.linalg.rq` does the work in one line.

Sanity checks worth running on your own output: $\mathbf{K}$ should be upper triangular with its
two diagonal entries (the focal lengths) of similar magnitude, and $\mathbf{R}$ should satisfy
$\mathbf{R}^T\mathbf{R} = \mathbf{I}$ with $\det \mathbf{R} = \pm 1$.


In [ ]:
print('Test for decomposing projection matrix:', verify(test_decompose_camera_matrix))

K, R = projection_matrix.decompose_camera_matrix(P)
print('K (normalized so K[2,2] = 1):\n', K / K[2, 2])
print('\nR:\n', R)
print('\nsanity checks:  R @ R.T == I ->', np.allclose(R @ R.T, np.eye(3)),
      '  det(R) = {:.4f}'.format(np.linalg.det(R)))


## Part 1.5: Calculating the camera center (6 pts)

Implement `calculate_camera_center(P, K, R)`, that takes as input the projection $\mathbf{P}$, intrinsic $\mathbf{K}$ and extrinsic ${}_c\mathbf{R}_w$ matrix, and returns the camera's position in **world**
coordinates. Return a flat array of shape `(3,)`.


In [ ]:
print('Test for calculating camera center:', verify(test_calculate_camera_center))

center = projection_matrix.calculate_camera_center(P, K, R)
print('Camera center in world coordinates:', center)
print('Distance to the scene centroid: {:.3f} world units'.format(
    np.linalg.norm(points_3d.mean(axis=0) - center)))


Now visualize the recovered camera center together with the world coordinate system.

In [ ]:
plot3dview(points_3d, center)


## Part 1.6: Your own images (5 pts)

Everything so far used correspondences we handed you. Now you will make your own.

A **fiducial** is an object of known size used as a dimensional reference. Pick something rigid you
can measure with a ruler, a thick textbook is ideal, and designate one of its corners as the
origin of your world coordinate system, with three edges as the $X$, $Y$, and $Z$ axes. Every point
you then pick on the object has a 3D coordinate you can measure directly. **Use consistent units
throughout** (millimetres or centimetres; the example below uses millimetres).

<img src="../data/projection_figure.png" width="600">

Take **two** photographs of the object from clearly different viewpoints. If you want to reuse the
same pair for Part 2, move the camera between shots rather than only rotating it.

A few things that will save you time:

- Pick points you can locate to within a pixel or two: corners beat edges, and edges beat the middle
  of a flat face.
- Pick at least 10, and spread them over all three dimensions. Points that are nearly coplanar make
  the estimation badly conditioned, which is the most common reason this part fails.
- Keep the ordering of `points_3d` and `points2d_img1`/`points2d_img2` identical — row $i$ of each
  must be the same physical point.


Load your two images. Replace these filenames with your own photographs.

In [ ]:
# TODO: replace these with the two photographs you took.
# The provided images are an example of workable framing, not your submission.
image1_path = '../data/new_book_img1.jpg'
image2_path = '../data/new_book_img2.jpg'

img1 = load_image(image1_path)
img2 = load_image(image2_path)
print('image 1: {} x {} px'.format(img1.shape[1], img1.shape[0]))
print('image 2: {} x {} px'.format(img2.shape[1], img2.shape[0]))


Measure your fiducial object and define its coordinate system, then fill in `points_3d`
with the 3D location of every point you will use.

In [ ]:
# TODO: the 3D coordinates of the points on your fiducial object, in your own
# units, measured from the origin corner you chose. One row per point: [X, Y, Z].
#
# The example below is for the provided textbook images (millimetres, 15 points).
# Replace it with your own measurements.
points_3d = np.array([
    [0, 0, 0],
])

raise NotImplementedError('Part 1.6: fill in points_3d with your own measurements')


Now find the 2D pixel location of each of those 3D points, in the same order, in each
image. Run the cell below and hover over the figure — the `x, y` coordinate under the cursor is
shown at the bottom right. The toolbar lets you zoom in for a more precise reading.

If the figure renders too small to click accurately, switch to an interactive backend by running
`%matplotlib widget` (or `%matplotlib notebook`) in a cell first.

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111)
ax.imshow(img1)
ax.set_title('image 1 - hover to read off pixel coordinates')
plt.show()


In [ ]:
# TODO: the 2D pixel location in IMAGE 1 of each 3D point above, in the same order.
points2d_img1 = np.array([
    [0, 0],
])

raise NotImplementedError('Part 1.6: fill in points2d_img1')


In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111)
ax.imshow(img2)
ax.set_title('image 2 - hover to read off pixel coordinates')
plt.show()


In [ ]:
# TODO: the 2D pixel location in IMAGE 2 of each 3D point, in the same order again.
points2d_img2 = np.array([
    [0, 0],
])

raise NotImplementedError('Part 1.6: fill in points2d_img2')


Save your measurements to `../results/` so the later cells can read them back.

In [ ]:
# Given: a few checks that catch the mistakes that otherwise surface much later
# as an optimizer that will not converge.
assert len(points_3d) == len(points2d_img1) == len(points2d_img2), (
    'the three arrays must have the same number of rows, in the same order; got '
    f'{len(points_3d)}, {len(points2d_img1)}, {len(points2d_img2)}')
assert len(points_3d) >= 10, f'use at least 10 points, got {len(points_3d)}'

for name, pts, im in [('image 1', points2d_img1, img1), ('image 2', points2d_img2, img2)]:
    h, w = im.shape[:2]
    inside = ((pts[:, 0] >= 0) & (pts[:, 0] < w) & (pts[:, 1] >= 0) & (pts[:, 1] < h))
    assert inside.all(), (
        f'{name}: these rows are outside the {w} x {h} image: '
        f'{np.flatnonzero(~inside).tolist()} -> {pts[~inside].tolist()}')

# Coplanar points make the estimation badly conditioned; this is the usual cause of a
# fit that looks converged but puts the camera somewhere impossible.
spread = points_3d.max(axis=0) - points_3d.min(axis=0)
print('3D extent along X, Y, Z:', spread)
if (spread == 0).any():
    print('WARNING: your points are coplanar (zero extent along one axis). '
          'Add points off that plane.')

np.savetxt(f'{RESULTS}/pts3d_fiducial.npy', points_3d)
np.savetxt(f'{RESULTS}/pts2d_image1.npy', points2d_img1)
np.savetxt(f'{RESULTS}/pts2d_image2.npy', points2d_img2)
print(f'saved {len(points_3d)} correspondences to {RESULTS}/')


## Part 1.7: Your own $\mathbf{K}$, $\mathbf{R}$, and $[\mathbf{I}|\mathbf{t}]$ estimates (5 pts)

Now build your own initial guess. As Part 1.3 warned, a poor one will not converge, so it is worth
reasoning about each piece rather than guessing:

- **$\mathbf{K}$** — the two diagonal entries are the focal length in pixels. For a phone camera a
  serviceable first guess is roughly the image width in pixels. The last column is the principal
  point, which is close to the image center, so use `(width/2, height/2)`. The cell below prints
  your image dimensions.
- **$\mathbf{R}^T$** — the camera's orientation relative to your world axes. Think about which way
  the camera was pointing when you took the shot.
- **$[\mathbf{I}|\mathbf{t}]$** — where the camera was, in **your** world units. If your textbook
  is 255 mm wide and you stood about half a metre away, $\mathbf{t}$ should have entries in the
  hundreds, not single digits.

The identity matrices below are placeholders, not a starting point: leaving them as they are will
not converge. Expect to iterate — change a value, rerun, look at the residual, repeat. 

**How to tell it worked.** A converged fit on a good set of correspondences gives a total residual
of a few hundred pixels or less across all your points (the provided example reaches about 330 for
15 points, i.e. roughly 20 px each). If you see residuals in the tens of thousands, or
`The maximum number of function evaluations is exceeded`, the optimizer did not converge and your
initial guess still needs work.


In [ ]:
# Load back the measurements for IMAGE 1
points_2d = np.loadtxt(f'{RESULTS}/pts2d_image1.npy')
points_3d = np.loadtxt(f'{RESULTS}/pts3d_fiducial.npy')
img = load_image(image1_path)

h, w = img.shape[:2]
print(f'image 1 is {w} x {h} px -> principal point is about ({w / 2:.0f}, {h / 2:.0f})')
print(f'a reasonable first guess for the focal length is around {w} px')


In [ ]:
# TODO: your initial guess for IMAGE 1. The identities below are placeholders and
# will NOT converge -- see the guidance above for how to pick each piece.
initial_guess_K = np.array([[1, 0, 0],
                            [0, 1, 0],
                            [0, 0, 1]])

initial_guess_R_T = np.array([[1, 0, 0],
                              [0, 1, 0],
                              [0, 0, 1]])

initial_guess_I_t = np.array([[1, 0, 0,  80],
                              [0, 1, 0, 290],
                              [0, 0, 1, -150]])

initial_guess_P = np.matmul(initial_guess_K, np.matmul(initial_guess_R_T, initial_guess_I_t))


Visualize your estimated camera pose relative to the world coordinate system. The RGB
axis colors correspond to XYZ. Be mindful of whether you should be passing `R` or `R.T` as the
rotation.

In [ ]:
plot3dview_with_coordinates(points_3d, -initial_guess_I_t[:, 3], initial_guess_R_T.T)


Run the optimization from your initial guess. Check the residual against the
guidance above before moving on.

In [ ]:
P1 = projection_matrix.estimate_camera_matrix(points_2d, points_3d, initial_guess_P)
print('The projection matrix for image 1 is\n', P1)

projected_2d_pts, residual = evaluate_points(P1, points_2d, points_3d)
print('Total residual: {:.3f} px over {} points ({:.2f} px per point)'.format(
    residual, len(points_2d), residual / len(points_2d)))

visualize_points_image(points_2d, projected_2d_pts, image1_path)


Now repeat the whole process for your second image: load its correspondences, build
a separate initial guess for it, and fit. The camera moved between shots, so the pose part of the
guess has to change too.

In [ ]:
# Load back the measurements for IMAGE 2
points_2d = np.loadtxt(f'{RESULTS}/pts2d_image2.npy')
points_3d = np.loadtxt(f'{RESULTS}/pts3d_fiducial.npy')
img = load_image(image2_path)

h, w = img.shape[:2]
print(f'image 2 is {w} x {h} px -> principal point is about ({w / 2:.0f}, {h / 2:.0f})')


In [ ]:
# TODO: your initial guess for IMAGE 2. The camera moved between shots, so this
# should not simply repeat the numbers you used for image 1.
initial_guess_K = np.array([[1, 0, 0],
                            [0, 1, 0],
                            [0, 0, 1]])

initial_guess_R_T = np.array([[1, 0, 0],
                              [0, 1, 0],
                              [0, 0, 1]])

initial_guess_I_t = np.array([[1, 0, 0,  80],
                              [0, 1, 0, 290],
                              [0, 0, 1, -160]])

initial_guess_P = np.matmul(initial_guess_K, np.matmul(initial_guess_R_T, initial_guess_I_t))


In [ ]:
plot3dview_with_coordinates(points_3d, -initial_guess_I_t[:, 3], initial_guess_R_T.T)


In [ ]:
P2 = projection_matrix.estimate_camera_matrix(points_2d, points_3d, initial_guess_P)
print('The projection matrix for image 2 is\n', P2)

projected_2d_pts, residual = evaluate_points(P2, points_2d, points_3d)
print('Total residual: {:.3f} px over {} points ({:.2f} px per point)'.format(
    residual, len(points_2d), residual / len(points_2d)))

visualize_points_image(points_2d, projected_2d_pts, image2_path)


## Part 1.8: Both camera poses in the world coordinate system (4 pts)

Decompose both projection matrices, recover both camera centers, and plot them together with the
scene points.

Check the result against the photographs you actually took: the two centers should sit roughly
where you were standing, at a plausible separation, and on the correct side of the object. If a
camera lands inside or behind your fiducial, something upstream is wrong — most often a
correspondence ordering mismatch or a sign error in the pose guess.

In [ ]:
K1, R1 = projection_matrix.decompose_camera_matrix(P1)
center_1 = projection_matrix.calculate_camera_center(P1, K1, R1)

K2, R2 = projection_matrix.decompose_camera_matrix(P2)
center_2 = projection_matrix.calculate_camera_center(P2, K2, R2)

print('camera 1 center:', center_1)
print('camera 2 center:', center_2)
print('baseline (distance between the two centers): {:.2f} world units'.format(
    np.linalg.norm(center_1 - center_2)))

plot3dview_2_cameras(points_3d, center_1, center_2, R1.T, R2.T)


## Part 1.9: How the camera center affects the result (5 pts)

"Changing the camera center" in two different ways, and each behave in completely different ways.

- **1.9.1 — the geometric effect.** Move the *actual* camera and re-project. This is pure geometry:
  no optimization is involved, and the result is predictable.
- **1.9.2 — the optimization effect.** Keep the true geometry fixed but change the *initial guess*
  handed to the optimizer. This is a numerical question about Levenberg–Marquardt, and the result
  is not predictable in the same way.

The following cells provide code demonstrating both effects. Carefully read the descriptions and code, run the cells, examine the results, and answer the corresponding questions.

### 1.9.1 The geometric effect of moving the camera

Take the $\mathbf{P}$ you estimated in Part 1.3, decompose it, slide the camera centre sideways,
rebuild $\mathbf{P}'$ from the same $\mathbf{K}$ and $\mathbf{R}$, and re-project the same 3D
points. **No optimization here** — you are simply asking what the camera would have seen from a
slightly different position.

Recall that a point at depth $Z$ in front of the camera lands at $u = f\,X_c / Z_c + u_0$. Keep
that in mind while you read the two tables the cell prints: the first doubles the size of the
sideways shift each time, and the second splits the same points into a nearer half and a farther
half.

(The camera is slid along its own sideways direction, so it means sideways from the camera's point
of view whichever way it happens to be facing. Moving it toward or away from the scene behaves
differently — worth trying if you are curious, but not part of the questions.)

In [ ]:
# 1.9.1  The GEOMETRIC effect of moving the camera. No optimization here.
#
# Given: this scaffold decomposes P, slides the camera centre sideways, rebuilds
#        P' = K [R | -R c'], and re-projects. Run it and read the two tables.
P_ref = P1
K_ref, R_ref = projection_matrix.decompose_camera_matrix(P_ref)
c_ref = projection_matrix.calculate_camera_center(P_ref, K_ref, R_ref)
pts3d_ref = np.loadtxt(f'{RESULTS}/pts3d_fiducial.npy')
base_proj = projection_matrix.projection(P_ref, pts3d_ref)

# The rows of R are the camera's own axes written in world coordinates, so this is
# "sideways from the camera's point of view" whatever direction it happens to face.
cam_right = R_ref[0, :] / np.linalg.norm(R_ref[0, :])
cam_fwd = R_ref[2, :] / np.linalg.norm(R_ref[2, :])
depth = np.abs((pts3d_ref - c_ref) @ cam_fwd)      # how far each point is from the camera


def slide_sideways(delta):
    """Re-project after sliding the true camera centre sideways by delta."""
    c_new = c_ref + delta * cam_right
    P_new = K_ref @ np.hstack([R_ref, (-R_ref @ c_new).reshape(3, 1)])
    return projection_matrix.projection(P_new / P_new[2, 3], pts3d_ref)


print('camera centre:', np.round(c_ref, 2))
print('\nhow far the image moves when the camera slides sideways')
print('{:>12}{:>16}{:>22}'.format('shift', 'displacement', 'ratio to row above'))

scale = max(1.0, depth.mean() / 100.0)
previous = None
for d in [0.02 * scale, 0.04 * scale, 0.08 * scale, 0.16 * scale]:
    moved = np.linalg.norm(slide_sideways(d) - base_proj, axis=1).mean()
    ratio = '-' if previous is None else '{:.2f}'.format(moved / previous)
    print('{:>12.3f}{:>16.2f}{:>22}'.format(d, moved, ratio))
    previous = moved

# Does everything in the image move by the same amount?
biggest = 0.16 * scale
moved = np.linalg.norm(slide_sideways(biggest) - base_proj, axis=1)
near = depth <= np.median(depth)

print('\nfor the largest shift, the same points split into near and far halves')
print('{:>20}{:>14}{:>16}'.format('half of the points', 'mean depth', 'displacement'))
print('{:>20}{:>14.1f}{:>16.2f}'.format('nearer', depth[near].mean(), moved[near].mean()))
print('{:>20}{:>14.1f}{:>16.2f}'.format('farther', depth[~near].mean(), moved[~near].mean()))

### 1.9.2 The effect of the initial guess on the optimization

Now change only the **initial guess** passed to `estimate_camera_matrix`, leaving the data
untouched. The true answer has not moved, so in principle the optimizer should find it every time.
Run the sweep below and see whether it does.

In [ ]:
# 1.9.2  The effect of the INITIAL GUESS on the optimization.
#
# Given: the correspondences never change, only the camera centre used to build the
# starting point. The right answer is in the same place every time.
import contextlib
import io

points_2d = np.loadtxt(f'{RESULTS}/pts2d_image1.npy')
points_3d = np.loadtxt(f'{RESULTS}/pts3d_fiducial.npy')

K_guess = np.array([[3000.0, 0.0, 2064.0],
                    [0.0, 3000.0, 1548.0],
                    [0.0, 0.0, 1.0]])
R_guess = np.eye(3)

print('{:>10}{:>18}{:>16}{:>14}'.format('guess c_x', 'total residual', 'px per point', 'verdict'))
for c_x in [-50, -25, 0, 25, 50, 100, 200, 400]:
    c = np.array([float(c_x), -250.0, 150.0])
    I_t = np.hstack([np.eye(3), (-R_guess @ c).reshape(3, 1)])
    guess_P = K_guess @ (R_guess @ I_t)
    with contextlib.redirect_stdout(io.StringIO()):      # hide the optimizer chatter
        P_try = projection_matrix.estimate_camera_matrix(points_2d, points_3d, guess_P)
    _, res = evaluate_points(P_try, points_2d, points_3d)
    per_pt = res / len(points_2d)
    print('{:>10}{:>18.1f}{:>16.2f}{:>14}'.format(
        c_x, res, per_pt, 'converged' if per_pt < 40 else 'DIVERGED'))


### Questions

Short answers, two or three sentences each is plenty.

1. In 1.9.1, what happened to the image displacement each time the shift doubled? Point at the part
   of $u = f\,X_c / Z_c + u_0$ that explains it.
2. Still in 1.9.1, which half of the points moved further, the nearer or the farther? Name an
   everyday situation where you have seen the same effect, and say in one sentence why this is
   useful if you want to recover depth from two views.
3. In 1.9.2 the data never changed, so the right answer stayed in the same place — but the
   optimizer did not always find it. Was "a closer starting guess gives a better fit" true? Say what
   that tells you about the method, and name one thing you would do about it in practice.

#### Your answers to 1.9

*1. What doubling the shift did, and why:*

*2. Near points against far points, and what it is good for:*

*3. Whether a closer guess gave a better fit, and what you would do about it:*

# Part 2: Fundamental Matrix Estimation

Part 1 needed known 3D points. The fundamental matrix needs none: given only 2D correspondences
between two views of a rigid scene, $\mathbf{F}$ captures the entire epipolar geometry relating
them.

Its defining property is that corresponding points satisfy
$\mathbf{x}_0^T \mathbf{F} \mathbf{x}_1 = 0$. Equivalently, $\mathbf{F}\mathbf{x}_1$ is a
**line** in the first image on which the match for $\mathbf{x}_1$ must lie. That is what reduces
correspondence search from a 2D problem to a 1D one, and it is the foundation for the stereo
matching you will do in Project 4.

You will estimate $\mathbf{F}$ by minimizing the symmetric point-to-line distance with SciPy's
least-squares optimizer
([documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html)).
The variables being optimized are the 9 entries of $\mathbf{F}$; the residuals are, for each
correspondence, how far each point falls from the epipolar line its partner induces.


## Part 2.1: Estimate the fundamental matrix (20 pts)

Implement `point_line_distance(line, point)` in `fundamental_matrix.py`:

\begin{align}
    d(\ell, \mathbf{x}) = \frac{au + bv + cw}{\sqrt{a^2 + b^2}}
\end{align}

where $\ell = (a, b, c)$ and $\mathbf{x} = (u, v, w)$ in homogeneous coordinates.

Note there is **no absolute value**: this is a *signed* distance, and that is deliberate. SciPy
squares the residuals itself, so the sign cancels.

In [ ]:
from unit_tests.test_fundamental_matrix import verify as verify_fm
from unit_tests.test_fundamental_matrix import TestFundamentalMatrix

test_fundamental_matrix_stereo = TestFundamentalMatrix()
TestFundamentalMatrix.setUp(test_fundamental_matrix_stereo)
print('test_point_line_distance():      '
      + verify_fm(test_fundamental_matrix_stereo.test_point_line_distance))
print('test_point_line_distance_zero(): '
      + verify_fm(test_fundamental_matrix_stereo.test_point_line_distance_zero))


Now implement `signed_point_line_errors()`, which applies that distance to every
correspondence.

The error is **symmetric**: each pair contributes two residuals, the distance from $\mathbf{x}_0$
to the line $\mathbf{F}\mathbf{x}_1$, and the distance from $\mathbf{x}_1$ to the line
$\mathbf{F}^T\mathbf{x}_0$:

\begin{align}
    \color{red}{d(\mathbf{F}\mathbf{x}_1, \mathbf{x}_0)}^2
  + \color{red}{d(\mathbf{F}^T \mathbf{x}_0, \mathbf{x}_1)}^2
\end{align}

Append them in that order — `d(Fx_1, x_0)` first, then `d(F'x_0, x_1)` — for each pair, so 9 points
produce a list of 18 floats. The unit tests check that ordering. Build this on top of
`point_line_distance()` rather than reimplementing the arithmetic inline.

You will also need to fill in the call to SciPy's optimizer in `optimize()` inside
`least_squares_fundamental_matrix.py`.

In [ ]:
from unit_tests.test_fundamental_matrix import (
    TestFundamentalMatrix2, TestFundamentalMatrix3)

print('TestFundamentalMatrix()  - real stereo pair:')
print('  test_signed_point_line_errors(): '
      + verify_fm(test_fundamental_matrix_stereo.test_signed_point_line_errors))
print('  test_least_squares_optimize():   '
      + verify_fm(test_fundamental_matrix_stereo.test_least_squares_optimize))

print('TestFundamentalMatrix2() - synthetic scene:')
test_fundamental_matrix_synthetic = TestFundamentalMatrix2()
TestFundamentalMatrix2.setUp(test_fundamental_matrix_synthetic)
print('  test_signed_point_line_errors(): '
      + verify_fm(test_fundamental_matrix_synthetic.test_signed_point_line_errors))
print('  test_least_squares_optimize():   '
      + verify_fm(test_fundamental_matrix_synthetic.test_least_squares_optimize))

print('TestFundamentalMatrix3() - real data:')
test_fundamental_matrix_real = TestFundamentalMatrix3()
TestFundamentalMatrix3.setUp(test_fundamental_matrix_real)
print('  test_signed_point_line_errors(): '
      + verify_fm(test_fundamental_matrix_real.test_signed_point_line_errors))
print('  test_least_squares_optimize():   '
      + verify_fm(test_fundamental_matrix_real.test_least_squares_optimize))


Run the cell below to estimate $\mathbf{F}$ for the provided image pair and draw the
epipolar lines. Each line should pass through, or very near, its corresponding point in the other
image.

In [ ]:
import least_squares_fundamental_matrix as ls

points_2d_pic_a = np.loadtxt('../data/pts2d-pic_a.txt')
points_2d_pic_b = np.loadtxt('../data/pts2d-pic_b.txt')
img_left = load_image('../data/pic_a.jpg')
img_right = load_image('../data/pic_b.jpg')

F = ls.solve_F(points_2d_pic_a, points_2d_pic_b)
print('F =\n', F)

draw_epipolar_lines(F, img_left, img_right, points_2d_pic_a, points_2d_pic_b)


## Part 2.2: Try it on your own images (15 pts)

Take two photographs yourself and estimate the fundamental matrix between them. Save them in
`../data/` and point the cell below at them.

What makes a good pair here:

- **Translate the camera between shots.** Question 2.3.1 asks you to explain why a pure rotation or
  a zoom will not work; take the pair in a way that avoids that failure.
- **Lots of texture at a range of depths.** Something in the foreground and something far behind it
  gives the epipolar geometry something to bite on.
- The example below (`my_image_0_new.jpg` / `my_image_1_new.jpg`) ships with the project so the
  cells run before you have your own. Replace it.

The picture below shows the kind of result you are aiming for.

<img src="../data/tokyo_tower_lines.png" width="700">

In [ ]:
# TODO: replace with your own pair.
my_img_left = load_image('../data/my_image_0_new.jpg')
my_img_right = load_image('../data/my_image_1_new.jpg')
print('left : {} x {} px'.format(my_img_left.shape[1], my_img_left.shape[0]))
print('right: {} x {} px'.format(my_img_right.shape[1], my_img_right.shape[0]))


Record corresponding points by hovering over each figure, exactly as you did in Part
1.6. Row $i$ of `my_image_0_pts` and row $i$ of `my_image_1_pts` must be the same physical feature.

You need **at least 9** points, because $\mathbf{F}$ has 9 entries (8 degrees of freedom after
scale). In practice 9 is not enough: every point you click carries a pixel or two of error, and with
exactly 9 the fit has no redundancy to average that out. **Aim for 18 or more.**

The cell after your points runs a sanity check on them before fitting.

In [ ]:
image_0 = plt.figure(figsize=(10, 8))
image_0_ax = image_0.add_subplot(111)
image_0_ax.imshow(my_img_left)
image_0_ax.set_title('image 0 - hover to read off pixel coordinates')
plt.show()


In [ ]:
# TODO: your corresponding points in IMAGE 0. At least 9; aim for 18 or more.
my_image_0_pts = np.array([
    [0, 0],
], dtype=float)

raise NotImplementedError('Part 2.2: fill in my_image_0_pts')


In [ ]:
image_1 = plt.figure(figsize=(10, 8))
image_1_ax = image_1.add_subplot(111)
image_1_ax.imshow(my_img_right)
image_1_ax.set_title('image 1 - hover to read off pixel coordinates')
plt.show()


In [ ]:
# TODO: the matching points in IMAGE 1, in the same order.
my_image_1_pts = np.array([
    [0, 0],
], dtype=float)

raise NotImplementedError('Part 2.2: fill in my_image_1_pts')


In [ ]:
# Given: catches the mistakes that are invisible in the final figure.
assert len(my_image_0_pts) == len(my_image_1_pts), (
    f'the two arrays must have the same length; got {len(my_image_0_pts)} '
    f'and {len(my_image_1_pts)}')
assert len(my_image_0_pts) >= 9, (
    f'F has 9 entries, so you need at least 9 correspondences; got '
    f'{len(my_image_0_pts)}')
if len(my_image_0_pts) < 18:
    print(f'NOTE: {len(my_image_0_pts)} points will work, but 18 or more gives the fit '
          'enough redundancy to average out your clicking error.')

for name, pts, im in [('image 0', my_image_0_pts, my_img_left),
                      ('image 1', my_image_1_pts, my_img_right)]:
    h, w = im.shape[:2]
    inside = (pts[:, 0] >= 0) & (pts[:, 0] < w) & (pts[:, 1] >= 0) & (pts[:, 1] < h)
    assert inside.all(), (
        f'{name}: rows {np.flatnonzero(~inside).tolist()} fall outside the {w} x {h} '
        f'image -> {pts[~inside].tolist()}. A single out-of-frame point can dominate '
        'the least-squares fit.')

print(f'{len(my_image_0_pts)} correspondences, all inside both images')


Estimate $\mathbf{F}$ for your pair and draw the epipolar lines.

In [ ]:
my_F = ls.solve_F(my_image_0_pts, my_image_1_pts)
print('my_F =\n', my_F)

# How well does the estimate actually satisfy the epipolar constraint?
x0 = np.hstack([my_image_0_pts, np.ones((len(my_image_0_pts), 1))])
x1 = np.hstack([my_image_1_pts, np.ones((len(my_image_1_pts), 1))])
dists = [abs(float(a @ my_F @ b)) / np.hypot(*(my_F @ b)[:2]) for a, b in zip(x0, x1)]
print('point-to-epipolar-line distance: mean {:.3f} px, max {:.3f} px'.format(
    np.mean(dists), np.max(dists)))

draw_epipolar_lines(my_F, my_img_left, my_img_right, my_image_0_pts, my_image_1_pts)


## Part 2.3: Questions (16 pts)

Answer each question in its own markdown cell below. Questions 1–4 are worth 3 points each,
questions 5–6 are worth 2 each.

1. Why can't you simply rotate the camera in place, or zoom, to produce your two images of the same
   scene?
2. Why does a point in one image map to an epipolar *line* in the other image, rather than to a
   point?
3. What happens to the epipoles and the epipolar lines when each camera center falls inside the
   other camera's field of view? Why?
4. What does it mean when all the epipolar lines come out horizontal and parallel across both
   images?
5. Why is the fundamental matrix defined only up to scale?
6. Why is the fundamental matrix rank 2? Then run the cell below, which reports the singular values
   of the $\mathbf{F}$ you estimated in Part 2.1, is *your* estimate actually rank 2? Explain what
   you see and what causes it.

Run this before answering question 6.

In [ ]:
# Given: the singular values of the F you estimated in Part 2.1.
singular_values = np.linalg.svd(F, compute_uv=False)
print('singular values of F:', singular_values)
print('ratio sigma_3 / sigma_1: {:.3e}'.format(singular_values[2] / singular_values[0]))
print('numerical rank (default tolerance):', np.linalg.matrix_rank(F))
print('is F skew-symmetric?', np.allclose(F, -F.T))


#### Answer 1

*Why a pure rotation or a zoom will not work.*


#### Answer 2

*Why a point maps to a line.*


#### Answer 3

*Epipoles and epipolar lines when the camera centers are in view.*


#### Answer 4

*What horizontal, parallel epipolar lines mean.*


#### Answer 5

*Why F is defined up to scale.*


#### Answer 6

*Why F is rank 2 in theory, and what the singular values of your estimate show.*


## Part 2.4: Robust estimation with RANSAC (10 pts, **extra credit**)

Everything so far assumed every correspondence was correct. In Part 2.2 that was nearly true,
because you clicked each point by hand. A real feature matcher is not so careful: 20-50% of its
matches are typically wrong, and least squares has no defence against even one of them. A single
bad match pulls the whole fit toward itself, because squaring a large error makes it dominate the
objective.

`data/points2_a.npy` and `data/points2_b.npy` hold **216 putative SIFT matches** between the two
Mount Rushmore images, produced automatically with no human filtering. Some are right, some are
not.

**RANSAC** (RANdom SAmple Consensus) handles this by not trusting any single fit. It repeatedly
draws a small random sample, fits a model to just that sample, and counts how many of the remaining
points agree with it within a tolerance. A sample made entirely of good matches produces a model
that many other points agree with; a sample containing an outlier does not. After enough draws, the
model with the largest agreeing set — its *inliers* — wins.

Here you will **use OpenCV's RANSAC to find the inliers**, then refit with **your own** estimator on
the clean subset. You are borrowing the robustness, not the estimator.

> **Note on allowed functions.** `cv2.findFundamentalMat` is permitted **in this section only**, and
> only to obtain the inlier mask. Using it in Part 2.1 or 2.2 would defeat those parts.

### Tasks
1. Fit on all 216 matches and see it fail — **2 pts**
2. Get an inlier mask from OpenCV's RANSAC, after checking its convention — **2 pts**
3. Refit your own `solve_F` on the inliers — **2 pts**
4. Sweep the threshold, check the iteration-count formula, and answer the questions — **4 pts**

A note on image scale: the shipped matches were computed on **half-resolution** copies of the
Rushmore images, so the given cell resizes the images by 0.5 before drawing. Drawing on the
full-size images would put every line in the wrong place.

In [ ]:
import two_view_data
from fundamental_matrix import point_line_distance

# Given: load the putative matches and set up a reporting helper.
#
# The matches were computed on HALF-RESOLUTION copies of the Rushmore images, so the
# images are resized to match before anything is drawn.
rushmore_a = cv2.resize(load_image('../data/2a_rushmore.jpg'), None, fx=0.5, fy=0.5)
rushmore_b = cv2.resize(load_image('../data/2b_rushmore.jpg'), None, fx=0.5, fy=0.5)

matches_a = np.load('../data/points2_a.npy')
matches_b = np.load('../data/points2_b.npy')
print(f'{len(matches_a)} putative matches; '
      f'images resized to {rushmore_a.shape[1]} x {rushmore_a.shape[0]} and '
      f'{rushmore_b.shape[1]} x {rushmore_b.shape[0]}')


def symmetric_epipolar_distance(F, pts_a, pts_b):
    """Largest of the two point-to-epipolar-line distances, per correspondence."""
    x_0s, x_1s = two_view_data.preprocess_data(pts_a.copy(), pts_b.copy())
    return np.array([max(abs(point_line_distance(F @ b, a)),
                         abs(point_line_distance(F.T @ a, b)))
                     for a, b in zip(x_0s, x_1s)])


def report(F, label, subset=None):
    """Print mean error and inlier count for a candidate F."""
    d = symmetric_epipolar_distance(F, matches_a, matches_b)
    on = d if subset is None else d[subset]
    print(f'{label:<40} mean {on.mean():7.3f} px   within 1 px: '
          f'{int((d < 1).sum()):3d}/{len(d)}')
    return d


def draw_sample(F, idx, n=20, seed=0):
    """Draw epipolar lines for a random handful of correspondences (216 is unreadable)."""
    rng = np.random.default_rng(seed)
    pick = rng.choice(np.asarray(idx), min(n, len(idx)), replace=False)
    draw_epipolar_lines(F, rushmore_a, rushmore_b, matches_a[pick], matches_b[pick])


### 2.4.1 Fit on everything (2 pts)

Run your own `solve_F` on all 216 matches and look at what you get.

In [ ]:
############################
# TODO: fit your own solve_F to ALL 216 matches, then report and draw it.
#       Use report(...) and draw_sample(..., np.arange(len(matches_a))).
raise NotImplementedError('2.4.1 needs to be implemented')
############################


### 2.4.2 Find the inliers with RANSAC (2 pts)

`cv2.findFundamentalMat(pts_a, pts_b, cv2.FM_RANSAC, threshold, confidence, maxIters)` returns a
fundamental matrix and a **mask** marking which correspondences it considers inliers. The mask is
what you are after.

**Check the convention before you use the result.** Different libraries disagree about which image
is "first", and the two choices differ by a transpose — which silently produces nonsense rather than
an error. Do not take anyone's word for it, including this handout: the cell below checks it against
the 20-point room pair from Part 2.1, where you already know the right answer. Report what you
found in your write-up.

In [ ]:
# Given: convention check against the room pair from Part 2.1, where the answer is known.
F_ours = ls.solve_F(points_2d_pic_a, points_2d_pic_b)
F_ocv, _ = cv2.findFundamentalMat(points_2d_pic_a, points_2d_pic_b, cv2.FM_8POINT)
unit = lambda M: M / np.linalg.norm(M)
print('max |difference| after normalizing both to unit Frobenius norm:')
print('   ours  vs  OpenCV      : {:.4f}'.format(np.abs(unit(F_ours) - unit(F_ocv)).max()))
print('   ours  vs  OpenCV.T    : {:.4f}'.format(np.abs(unit(F_ours) - unit(F_ocv.T)).max()))

############################
# TODO: run RANSAC on the 216 matches to get the inlier mask.
#   - cv2.findFundamentalMat(matches_a, matches_b, cv2.FM_RANSAC, threshold,
#                            confidence, maxIters) -> (F, mask)
#   - start with threshold = 1.0, confidence = 0.999, maxIters = 10000
#   - apply whatever the check above tells you about the convention
#   - store the boolean inlier mask in `inliers`
raise NotImplementedError('2.4.2 needs to be implemented')
############################

print(f'\nRANSAC kept {int(inliers.sum())} of {len(matches_a)} matches as inliers')


### 2.4.3 Refit with your own estimator (2 pts)

RANSAC's own fit came from a small random sample. Now that you know *which* matches are
trustworthy, refit your `solve_F` using all of the inliers and none of the outliers.

In [ ]:
############################
# TODO: refit your own solve_F using only the inliers, then report and draw.
#       Pass subset=inliers to report(...) so the mean is taken over the same points.
raise NotImplementedError('2.4.3 needs to be implemented')
############################


### 2.4.4 Parameters and interpretation (4 pts)

The inlier threshold and the iteration count are the two parameters that decide whether RANSAC
works. The cell below sweeps the threshold and computes the theoretical iteration requirement; run
it, then answer the questions.

The number of samples needed for a probability $p$ of drawing at least one all-inlier sample is

$$N = \frac{\log(1-p)}{\log\left(1-w^{s}\right)}$$

where $w$ is the fraction of correspondences that are inliers and $s$ is the sample size.

In [ ]:
# Given: threshold sweep, iteration formula, and rank check. Run it, then answer below.
print('threshold sweep (confidence 0.999, maxIters 10000)')
print('{:>8}{:>12}{:>22}'.format('thr (px)', 'inliers', 'mean dist on inliers'))
for thr in [0.5, 1.0, 2.0, 3.0, 5.0, 10.0]:
    F_t, m_t = cv2.findFundamentalMat(matches_a, matches_b, cv2.FM_RANSAC, thr, 0.999, 10000)
    m_t = m_t.ravel().astype(bool)
    d_t = symmetric_epipolar_distance(F_t.T, matches_a, matches_b)
    print('{:>8.1f}{:>8d}/{:<3d}{:>22.3f}'.format(thr, int(m_t.sum()), len(matches_a),
                                                  d_t[m_t].mean()))


def ransac_iterations(prob_success, sample_size, inlier_rate):
    """Samples needed for prob_success odds of drawing one all-inlier sample."""
    return int(np.ceil(np.log(1 - prob_success)
                       / np.log(1 - inlier_rate ** sample_size)))


print('\niterations N needed for p = 0.999')
print('{:>12}{:>10}{:>10}{:>10}'.format('inlier rate', 's = 7', 's = 8', 's = 9'))
for w in [0.3, 0.5, 0.7, 0.9]:
    print('{:>12.1f}{:>10d}{:>10d}{:>10d}'.format(
        w, *[ransac_iterations(0.999, s, w) for s in (7, 8, 9)]))

observed_w = inliers.mean()
print(f'\nobserved inlier rate here: {observed_w:.2f} '
      f'-> N = {ransac_iterations(0.999, 8, observed_w)} at s = 8')

print('\ndoes maxIters match the prediction?')
for max_it in [5, 20, 100, 1000, 10000]:
    cv2.setRNGSeed(0)
    _, m_i = cv2.findFundamentalMat(matches_a, matches_b, cv2.FM_RANSAC, 1.0, 0.999, max_it)
    print(f'   maxIters {max_it:>6} -> {int(m_i.sum()):3d} inliers')

print('\nrank check (compare with Question 2.3.6)')
for label, F_k in [('solve_F on all matches', F_all),
                   ("OpenCV's RANSAC F", F_ransac),
                   ('solve_F refit on inliers', F_refit)]:
    sv = np.linalg.svd(F_k, compute_uv=False)
    print('   {:<26} sigma3/sigma1 = {:.2e}   numerical rank {}'.format(
        label, sv[2] / sv[0], np.linalg.matrix_rank(F_k)))


### Questions

1. **The threshold.** What does it trade off? Describe the failure mode at each end of your sweep, 
   what goes wrong at 0.5 px, and what goes wrong at 10 px?
2. **The iteration count.** Using the printed table, explain why $N$ grows so steeply as the inlier
   rate drops, and why it grows with sample size. Does the `maxIters` experiment agree with the
   formula? This project's estimator needs at least 9 points, what does that cost compared with a
   7-point minimal solver?
3. **Rank.** Compare the singular values of all three matrices. Did RANSAC fix the rank problem you
   found in Question 2.3.6? Explain what you see, and say what that tells you about the relationship
   between the two defects.
4. **The failed fit.** Look at the epipolar lines from 2.4.1. They are not merely inaccurate, they
   have a specific, recognizable structure. Describe it, and explain geometrically what the
   outliers did to the estimate.

#### Your answers to 2.4

*1 — the threshold trade-off:*

*2 — the iteration count:*

*3 — rank:*

*4 — the structure of the failed fit:*


# Part 3: AI-Use Disclosure (required)

See the syllabus for the details of the course AI policy. 

Answer the questions below. If you didn't use any AI tools, write "None"
for the first question, skip the second. 

### Your disclosure

**1. Which AI tools did you use, if any?**
*(Name the model, the version, the subscription plan, code editor, and any related AI tools. Write "None" if you used none.)*


**2. What did you use them for, and which parts of this notebook did they influence?**
*Be specific about both.
List the cells, functions, or written answers involved.*

---

**Statement.** *By submitting this notebook you confirm that the code and written answers in it are
your own work, that you did not use AI tools to solve any part of this assignment, and that any
other AI use is fully described above.*


---

**Before you submit:** restart the kernel, run every cell from top to bottom, confirm that every
figure and printed result is visible, save the notebook, then run
`python zip_submission.py --uid <your_uid>` from the project folder.